### VERIFIER AGENT


Retrived products ---> Review Summaries ---> Ranking Agent ---> Final Ranked Products

In [1]:
import os
import sys

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
from IPython.display import display
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

from Agents.retrieval_agent import RetrievalAgent, ProductQuery
from Agents.review_agent import ReviewAgent
from Agents.ranking_agent import RankingAgent
from Agents.verifier_agent import VerifierAgent


C:\Users\srush\AppData\Local\Temp\ipykernel_37304\1736237207.py:14: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
load_dotenv()

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


In [3]:
vector_db = FAISS.load_local(
    "../Data/Cleaned/faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

reviews_df = pd.read_parquet(
    "../Data/Cleaned/reviews_sample.parquet"
)

In [4]:
retrieval_agent = RetrievalAgent(vector_db)

review_agent = ReviewAgent(
    llm,
    reviews_df
)

ranking_agent = RankingAgent(llm)

verifier_agent = VerifierAgent(llm)

In [12]:
query = ProductQuery(
    product_type="phone",
    brand="samsung",
    budget="50000",
    features=[
        "good camera",
        "fast performance"
    ]
)

In [13]:
retrieved_products = retrieval_agent.retrieve(query)

print(
    f"Retrieved {len(retrieved_products)} products"
)

if not retrieved_products:
    raise ValueError(
        "No products were retrieved."
    )

Retrieved 10 products


In [14]:
products = []

for doc, retrieval_score in retrieved_products:

    parent_asin = doc.metadata["parent_asin"]

    try:
        review = review_agent.summarize_reviews(
            parent_asin
        )

    except Exception as e:

        print(
            f"Review summarisation failed for "
            f"{parent_asin}: {e}"
        )

        # Use empty fallback only when summarisation fails
        review = {}

    products.append({

        "parent_asin": parent_asin,

        "title": doc.metadata["title"],

        "price": doc.metadata.get(
            "price"
        ),

        "currency": doc.metadata.get(
            "currency"
        ),

        "retrieval_score": float(
            doc.metadata.get(
                "retrieval_score",
                retrieval_score
            ) or 0
        ),

        "average_rating": float(
            doc.metadata.get(
                "average_rating",
                0
            ) or 0
        ),

        "rating_number": int(
            doc.metadata.get(
                "rating_number",
                0
            ) or 0
        ),

        "review_count": int(
            review.get(
                "review_count",
                0
            ) or 0
        ),

        "overall_sentiment": review.get(
            "overall_sentiment",
            "unknown"
        ),

        "pros": review.get(
            "pros",
            []
        ) or [],

        "cons": review.get(
            "cons",
            []
        ) or [],

        "recommended_for": review.get(
            "recommended_for",
            ""
        ),

        "avoid_if": review.get(
            "avoid_if",
            ""
        ),

        "review_summary": review.get(
            "summary",
            ""
        )

    })


if not products:
    raise ValueError(
        "No products available for ranking."
    )



In [15]:
ranking = ranking_agent.rank_products(
    query,
    products
)

In [16]:
verification = verifier_agent.verify(
    query=query,
    ranking_output=ranking,
    original_products=products
)

In [17]:
print("=" * 80)
print("FINAL VERIFICATION")
print("=" * 80)

print(
    "Overall Status:",
    verification.overall_status
)

print(
    "Confidence:",
    verification.confidence
)

print(
    "Recommended Product:",
    verification.recommended_product_title
)

print(
    "Recommended ASIN:",
    verification.recommended_product_asin
)

print()
print("Summary")
print("-" * 40)

print(
    verification.summary
)

FINAL VERIFICATION
Overall Status: passed_with_warnings
Confidence: medium
Recommended Product: Samsung Galaxy Mega 6.3 I9200 8GB Unlocked GSM Dual-Core Android Smartphone w/ 8 MP Camera - Black
Recommended ASIN: B00D93LOY6

Summary
----------------------------------------
The recommended product is the **Samsung Galaxy Mega 6.3 I9200 8GB Unlocked GSM Dual-Core Android Smartphone**. It passed verification with warnings, primarily due to the unavailability of its price, which means budget compliance could not be verified. While it fully matches the requested features of a good camera and fast performance, it does not meet the budget criteria. The overall status of the verification is "passed_with_warnings," and the confidence level is medium.


In [18]:
for product in verification.verified_products:

    print("=" * 80)

    print(
        "Rank:",
        product.rank
    )

    print(
        "Title:",
        product.title
    )

    print(
        "Status:",
        product.status
    )

    print(
        "Brand Match:",
        product.brand_match
    )

    print(
        "Product Type Match:",
        product.product_type_match
    )

    print(
        "Feature Status:",
        product.feature_status
    )

    print(
        "Feature Match Ratio:",
        f"{product.feature_match_ratio:.0%}"
    )

    print(
        "Matched Features:",
        product.matched_features
    )

    print(
        "Missing Features:",
        product.missing_features
    )

    print(
        "Budget Verifiable:",
        product.budget_verifiable
    )

    print(
        "Budget Match:",
        product.budget_match
    )

    print(
        "Evidence Strength:",
        product.evidence_strength
    )

    print(
        "Reason:",
        product.verification_reason
    )

    if product.warnings:

        print("\nWarnings")

        for warning in product.warnings:
            print("-", warning)

    print()

# ----------------------------------------------------
# Verification Summary Table
# ----------------------------------------------------

verification_df = pd.DataFrame([
    {
        "Rank": p.rank,
        "Title": p.title,
        "Status": p.status,
        "Brand Match": p.brand_match,
        "Product Type Match": p.product_type_match,
        "Feature Status": p.feature_status,
        "Feature Match Ratio": p.feature_match_ratio,
        "Matched Features": ", ".join(p.matched_features),
        "Missing Features": ", ".join(p.missing_features),
        "Budget Verifiable": p.budget_verifiable,
        "Budget Match": p.budget_match,
        "Evidence Strength": p.evidence_strength
    }
    for p in verification.verified_products
])

print("=" * 80)
print("VERIFICATION SUMMARY TABLE")
print("=" * 80)

display(verification_df)

Rank: 1
Title: Samsung Galaxy Mega 6.3 I9200 8GB Unlocked GSM Dual-Core Android Smartphone w/ 8 MP Camera - Black
Status: passed_with_warning
Brand Match: True
Product Type Match: True
Feature Status: full_match
Feature Match Ratio: 100%
Matched Features: ['good camera', 'fast performance']
Missing Features: []
Budget Verifiable: True
Budget Match: False
Evidence Strength: strong
Reason: Brand match: True. Product type match: True. Feature status: full_match. Feature match ratio: 1.00. Evidence strength: strong. Price available: True. Budget requested: True. Budget match: False.

Warnings
- Budget compliance could not be verified because the product price was unavailable.

Rank: 2
Title: Samsung Galaxy S5 SM-G900H Factory Unlocked Cellphone, International Version, Black
Status: passed
Brand Match: True
Product Type Match: True
Feature Status: full_match
Feature Match Ratio: 100%
Matched Features: ['good camera', 'fast performance']
Missing Features: []
Budget Verifiable: True
Budget Ma

,Rank,Title,Status,Brand Match,Product Type Match,Feature Status,Feature Match Ratio,Matched Features,Missing Features,Budget Verifiable,Budget Match,Evidence Strength
0,1,Samsung Galaxy Mega 6.3 I9200 8GB Unlocked GSM...,passed_with_warning,True,True,full_match,1.0,"good camera, fast performance",,True,False,strong
1,2,Samsung Galaxy S5 SM-G900H Factory Unlocked Ce...,passed,True,True,full_match,1.0,"good camera, fast performance",,True,True,strong
2,3,"Samsung Galaxy A52 (5G) 128GB A526U 6.5"" Displ...",passed,True,True,full_match,1.0,"good camera, fast performance",,True,True,strong
3,4,Samsung Galaxy Rugby Pro 4G LTE I547 Unlocked ...,passed,True,True,full_match,1.0,"good camera, fast performance",,True,True,strong
4,5,"Samsung Galaxy S22 Smartphone, Factory Unlocke...",passed,True,True,full_match,1.0,"good camera, fast performance",,True,True,moderate
5,6,Samsung Galaxy S10 G973U Unlocked 128GB - Flam...,passed_with_warning,True,True,partial_match,0.5,good camera,fast performance,True,False,strong
6,7,"Samsung Galaxy S21 FE 5G Cell Phone, Factory U...",passed_with_warning,True,True,partial_match,0.5,good camera,fast performance,True,True,moderate
7,8,SAMSUNG Galaxy S21 FE 5G SM-G990U 256GB Factor...,passed_with_warning,True,True,no_match,0.0,,"good camera, fast performance",True,True,moderate
8,9,SAMSUNG Galaxy S10 Factory Unlocked Phone with...,passed_with_warning,True,True,no_match,0.0,,"good camera, fast performance",True,False,strong
9,10,"SAMSUNG Galaxy S21 Ultra 5G, 128GB, Phantom Bl...",passed_with_warning,True,True,no_match,0.0,,"good camera, fast performance",True,True,moderate
